# GGPONC 2.0 als Test-Set — Zero-Shot-Evaluation

Testet deine bereits trainierten Modelle (Baseline und +synth, je 4 Seeds) **ohne
Nachtraining** auf GGPONC (deutsche Onkologie-**Leitlinien**). Geladen über den
**BigBIO-Loader** `bigbio/ggponc2`. Gewertet wird per-Typ-F1 für **Medikation** und
**Diagnose**.

**Zwei GGPONC-Besonderheiten, die hier berücksichtigt sind:**
- **Kurze Spans:** wir nehmen die Config `..._fine_short_...` — feingranulare Klassen mit
  *kurzen* Spans, näher an der Granularität deines GPTNERMED-Modells (die `long`-Variante
  würde v. a. Grenz-Fehler messen).
- **Zusätzlich Overlap-Wertung:** neben exaktem Match rechnen wir auch eine *partielle*
  (Overlap-)F1 — fairer, wenn Grenzen leicht abweichen.

**Wichtig:** lokal im Env `spacy-test` ausführen; GGPONC nicht in die Cloud laden (DUA).

## 1) Konfiguration — Pfade & Config anpassen

Braucht `datasets` (mit `trust_remote_code`), ggf. `pip install bigbio`. Ist dein
`datasets` zu neu und lehnt Skript-Loader ab, eine ältere Version verwenden.

In [1]:
from pathlib import Path

# GGPONC-Daten (nach DUA lokal abgelegt). data_dir wie vom BigBIO-Loader erwartet:
GGPONC_DATA_DIR = "pfad/zu/ggponc/data/v2.0_2022_03_24"
BIGBIO_CONFIG   = "ggponc2_fine_short_bigbio_kb"   # fein + KURZE Spans (fairer fuer dein Modell)
SPLIT           = "test"                            # zero-shot: Test-Split (oder "train"/"validation")

GGPONC_SPACY = "meaningful_modification/data/ggponc_test.spacy"

# Deine trainierten Modelle: jeweils der model-best-Ordner. PFADE/NAMEN ANPASSEN!
MODELS = {
    "baseline": [
        "related_work/models/gbert-large-seed-42-model-best",
    ],
    "+synth": [
        "meaningful_modification/models/gbert-large-syn-seed42-model-best",
    ],
}

# GGPONC-Entitaetstyp -> dein Label. Genaue Strings in Zelle 2 verifizieren!
LABEL_MAP = {
    "Clinical_Drug": "Medikation",
    "Diagnosis_or_Pathology": "Diagnose",
    # ignoriert: Other_Finding, Nutrient_or_Body_Substance, External_Substance,
    #            Therapeutic, Diagnostic
}
EVAL_LABELS = ["Medikation", "Diagnose"]

# Zum schnellen Antesten z.B. 300; fuer den echten Lauf None
MAX_DOCS = None

## 2) GGPONC laden & Entitätstypen prüfen

Zeigt die verfügbaren Config-Namen und die tatsächlichen Typ-Strings — damit gleichst du
`BIGBIO_CONFIG` und `LABEL_MAP` oben ab.

In [2]:
from datasets import load_dataset, get_dataset_config_names

# Verfuegbare Configs (falls der Name oben nicht passt)
try:
    print("Configs:", get_dataset_config_names("bigbio/ggponc2", trust_remote_code=True))
except Exception as e:
    print("Config-Liste nicht abrufbar:", e)

ds = load_dataset(
    "bigbio/ggponc2",
    data_dir=GGPONC_DATA_DIR,
    name=BIGBIO_CONFIG,
    trust_remote_code=True,
)
print(ds)

split = ds[SPLIT] if SPLIT in ds else ds[list(ds.keys())[0]]

# Vorhandene Entitaetstypen -> mit LABEL_MAP abgleichen
types = sorted({ent["type"] for ex in split for ent in ex["entities"]})
print("Entitaetstypen im Datensatz:", types)
print("Beispiel-Entities:", split[0]["entities"][:5])

Configs: ['ggponc2_fine_long_source', 'ggponc2_fine_short_source', 'ggponc2_coarse_long_source', 'ggponc2_coarse_short_source', 'ggponc2_fine_long_bigbio_kb', 'ggponc2_fine_short_bigbio_kb', 'ggponc2_coarse_long_bigbio_kb', 'ggponc2_coarse_short_bigbio_kb']


Generating train split: 0 examples [00:00, ? examples/s]

DatasetGenerationError: An error occurred while generating the dataset

## 3) In `.spacy` konvertieren (nur Medikation/Diagnose)

In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans


def example_to_doc(nlp, example):
    """BigBIO-KB-Beispiel -> spaCy Doc; Text ueber absolute Offsets rekonstruiert."""
    passages = example["passages"]
    end = max((off[1] for p in passages for off in p["offsets"]), default=0)
    buf = [" "] * end
    for p in passages:
        for txt, off in zip(p["text"], p["offsets"]):
            start = off[0]
            for i, ch in enumerate(txt):
                if start + i < end:
                    buf[start + i] = ch
    text = "".join(buf)

    doc = nlp.make_doc(text)
    spans = []
    for ent in example["entities"]:
        label = LABEL_MAP.get(ent["type"])
        if not label or not ent["offsets"]:
            continue
        s = ent["offsets"][0][0]
        e = ent["offsets"][-1][1]
        span = doc.char_span(s, e, label=label, alignment_mode="expand")
        if span is not None:
            spans.append(span)
    doc.ents = filter_spans(spans)
    return doc


nlp_blank = spacy.blank("de")
doc_bin = DocBin()
n_docs = n_ent = 0
for example in split:
    if MAX_DOCS is not None and n_docs >= MAX_DOCS:
        break
    doc = example_to_doc(nlp_blank, example)
    doc_bin.add(doc)
    n_docs += 1
    n_ent += len(doc.ents)

Path(GGPONC_SPACY).parent.mkdir(parents=True, exist_ok=True)
doc_bin.to_disk(GGPONC_SPACY)
print(f"{n_docs} Dokumente, {n_ent} Entitaeten (Medikation/Diagnose) -> {GGPONC_SPACY}")

## 4) Beide Modelle evaluieren — exakt **und** Overlap

In [ ]:
import re
import spacy
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.scorer import Scorer


def spans_of(doc, labels):
    return [(e.start_char, e.end_char, e.label_) for e in doc.ents if e.label_ in labels]


def overlaps(a, b):
    return not (a[1] <= b[0] or b[1] <= a[0])


def evaluate_model(model_path, spacy_path, labels, max_docs=None):
    nlp = spacy.load(model_path)
    golds = list(DocBin().from_disk(spacy_path).get_docs(nlp.vocab))
    if max_docs is not None:
        golds = golds[:max_docs]

    examples = []
    tp = {l: 0 for l in labels}
    fp = {l: 0 for l in labels}
    fn = {l: 0 for l in labels}
    for gold in golds:
        pred = nlp(gold.text)
        examples.append(Example(pred, gold))
        gset = spans_of(gold, labels)
        pset = spans_of(pred, labels)
        used = set()
        for p in pset:
            hit = False
            for j, g in enumerate(gset):
                if j not in used and p[2] == g[2] and overlaps(p, g):
                    used.add(j)
                    hit = True
                    break
            tp[p[2]] += 1 if hit else 0
            fp[p[2]] += 0 if hit else 1
        for j, g in enumerate(gset):
            if j not in used:
                fn[g[2]] += 1

    exact = Scorer().score(examples).get("ents_per_type", {})
    overlap = {}
    for l in labels:
        prec = tp[l] / (tp[l] + fp[l]) if (tp[l] + fp[l]) else 0.0
        rec = tp[l] / (tp[l] + fn[l]) if (tp[l] + fn[l]) else 0.0
        f = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        overlap[l] = {"p": prec, "r": rec, "f": f}
    return exact, overlap


records = []
for condition, paths in MODELS.items():
    for model_path in paths:
        match = re.search(r"seed[-_]?(\d+)", model_path)
        seed = int(match.group(1)) if match else -1

        exact, overlap = evaluate_model(model_path, GGPONC_SPACY, EVAL_LABELS, MAX_DOCS)
        for label in EVAL_LABELS:
            records.append({
                "condition": condition, "seed": seed, "label": label,
                "f_exact": exact.get(label, {}).get("f", 0.0),
                "f_overlap": overlap.get(label, {}).get("f", 0.0),
            })
        print(f"[fertig] {condition} seed {seed}: " + ", ".join(
            f"{l} exact={exact.get(l, {}).get('f', 0):.3f}/overlap={overlap.get(l, {}).get('f', 0):.3f}"
            for l in EVAL_LABELS))

## 5) Auswertung: Baseline vs. +synth (mean über Seeds)

In [ ]:
import pandas as pd

df = pd.DataFrame(records)

for metric in ["f_exact", "f_overlap"]:
    summary = (df.groupby(["label", "condition"])[metric]
                 .agg(mean="mean", std="std").reset_index())
    tbl = summary.pivot(index="label", columns="condition", values="mean").round(4)
    tbl["delta (+synth - baseline)"] = (tbl.get("+synth") - tbl.get("baseline")).round(4)
    print(f"=== {metric} (Mittelwert ueber Seeds) ===")
    print(tbl)
    print()

df  # Rohwerte pro Modell/Seed